# Complete Redis Functionality Test

This notebook verifies Redis for the Data Governance Copilot end to end:

- Environment and Redis configuration
- Live Redis connection and server health
- `cache_set`, `cache_get`, TTL, JSON round trip
- Deterministic cache keys via `make_key`
- `@cached_node` miss/hit behavior
- Optional graph-level cache test using the same path as the UI
- Redis key inspection and cleanup

Run it from the project root. If you run locally, Redis should be reachable at `localhost:6379`. If you run inside Docker Compose, the app container should use `redis:6379`.

In [ ]:
from pathlib import Path
import os
import sys
import time
import json

# Resolve project root from notebook/phase or current working directory.
cwd = Path.cwd().resolve()
project_root = cwd
for candidate in [cwd, *cwd.parents]:
    if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
        project_root = candidate
        break

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("project_root:", project_root)
print("src_path:", src_path)

## 1. Load Config

This confirms the environment values that the app will use. Passwords are redacted.

In [ ]:
from config.settings import AppConfig

cfg = AppConfig().redis

print("REDIS_ENABLED:", cfg.enabled)
print("REDIS_HOST:", cfg.host)
print("REDIS_PORT:", cfg.port)
print("REDIS_DB:", cfg.db)
print("REDIS_PASSWORD_SET:", bool(cfg.password))
print("REDIS_URL:", cfg.url.replace(cfg.password, "<redacted>") if cfg.password else cfg.url)

## 2. Live Redis Connection

If this cell reports `client: False`, the app will fall back to in-memory caching and Redis will stay empty.

In [ ]:
import core.cache as cache
from core.cache import get_client

# Reset module singleton so this notebook tests the current config cleanly.
cache._client = None
client = get_client(cfg)

print("client:", bool(client))
if client:
    print("ping:", client.ping())
    print("redis_version:", client.info().get("redis_version"))
else:
    print("Redis is unavailable. Fix host/port/container config before continuing.")

## 3. Test Namespace Cleanup

The notebook uses only `nb_redis_test:*` keys for direct tests and cleans them up safely.

In [ ]:
TEST_PREFIX = "nb_redis_test"

def redis_keys(pattern):
    if not client:
        return []
    return sorted(client.scan_iter(pattern))

def delete_pattern(pattern):
    if not client:
        return 0
    keys = list(client.scan_iter(pattern))
    return client.delete(*keys) if keys else 0

deleted = delete_pattern(f"{TEST_PREFIX}:*")
print("deleted_old_test_keys:", deleted)
print("dbsize_before:", client.dbsize() if client else "no redis")

## 4. Direct Cache Round Trip

This tests your project helpers: `cache_set` writes JSON with TTL, and `cache_get` reads it back as Python data.

In [ ]:
from core.cache import cache_set, cache_get

key = f"{TEST_PREFIX}:roundtrip"
payload = {
    "agent": "redis_notebook",
    "success": True,
    "metrics": {"retention": 81.65, "churn": 18.35},
    "items": ["a", "b", "c"],
}

cache_set(client, key, payload, ttl=120)
readback = cache_get(client, key)

print("readback:", readback)
print("ttl:", client.ttl(key) if client else "no redis")
assert readback == payload
assert client.ttl(key) > 0
print("PASS: direct cache round trip")

## 5. TTL Expiry

This proves Redis expires cache values after the configured TTL.

In [ ]:
short_key = f"{TEST_PREFIX}:ttl"
cache_set(client, short_key, {"expires": True}, ttl=2)

print("immediate:", cache_get(client, short_key))
print("ttl_initial:", client.ttl(short_key) if client else "no redis")
time.sleep(3)
print("after_sleep:", cache_get(client, short_key))
print("exists_after_sleep:", client.exists(short_key) if client else "no redis")

assert cache_get(client, short_key) is None
print("PASS: TTL expiry")

## 6. Deterministic Cache Keys

Your graph cache keys are based on query, data products, and time range.

In [ ]:
from core.cache import make_key

k1 = make_key(
    "information_agent",
    query="Why did retention drop last month?",
    data_products=["retention"],
    time_range="last_month",
)
k2 = make_key(
    "information_agent",
    time_range="last_month",
    data_products=["retention"],
    query="Why did retention drop last month?",
)
k3 = make_key(
    "information_agent",
    query="Why did retention drop last month?",
    data_products=["bookings"],
    time_range="last_month",
)

print("k1:", k1)
print("k2:", k2)
print("k3:", k3)
assert k1 == k2
assert k1 != k3
print("PASS: deterministic keys")

## 7. Decorator Miss and Hit

This tests the same `@cached_node` wrapper used by `information_node`, `knowledge_node`, and `metadata_node`.

In [ ]:
from core.cache import cached_node

decorator_prefix = f"{TEST_PREFIX}:decorated_node"
delete_pattern(f"{decorator_prefix}:*")

calls = {"count": 0}

@cached_node(decorator_prefix, ttl=120)
def expensive_node(state):
    calls["count"] += 1
    return {
        "agent_results": [
            {"agent": "notebook_agent", "success": True, "summary": "computed"}
        ]
    }

state = {
    "query": "same question",
    "data_products": ["retention"],
    "time_range": "last_month",
}

first = expensive_node(state)
second = expensive_node(state)

print("calls:", calls["count"])
print("first:", first)
print("second:", second)
print("decorator_keys:", redis_keys(f"{decorator_prefix}:*"))

assert calls["count"] == 1
assert first == second
print("PASS: decorator miss then hit")

## 8. Inspect Existing App Cache Keys

The UI creates keys like `information_agent:*`, `knowledge_agent:*`, and `metadata_agent:*`. Redis logs will not show normal `GET` or `SETEX`; inspect the keyspace instead.

In [ ]:
for pattern in ["information_agent:*", "knowledge_agent:*", "metadata_agent:*", f"{TEST_PREFIX}:*"]:
    keys = redis_keys(pattern)
    print(f"\n{pattern} -> {len(keys)} key(s)")
    for k in keys[:20]:
        print(" ", k, "ttl=", client.ttl(k) if client else "no redis")

## 9. Optional Graph-Level Cache Test

This runs the LangGraph path used by the UI. It can call your LLM provider for intent classification, so skip this if you only want Redis unit tests.

Expected behavior:
- First run: cache miss for read agents, then Redis keys are set.
- Second run with same query/time range/products: cache hit for cached nodes.
- `capacity_node`, `rule_node`, `auto_ticket_node`, and `synthesizer_node` are not cached.

In [ ]:
RUN_GRAPH_TEST = False

if RUN_GRAPH_TEST:
    from graph.graph import copilot_graph
    from graph.state import initial_state

    graph_query = "Why did retention drop last month?"
    thread_id = "redis-notebook-test"
    graph_cfg = {"configurable": {"thread_id": thread_id}}

    state = initial_state(
        query=graph_query,
        thread_id=thread_id,
        user_id="notebook-user",
        time_range="last_month",
    )

    t0 = time.time()
    first = copilot_graph.invoke(state, config=graph_cfg)
    first_ms = round((time.time() - t0) * 1000, 2)

    t1 = time.time()
    second = copilot_graph.invoke(state, config=graph_cfg)
    second_ms = round((time.time() - t1) * 1000, 2)

    print("first_ms:", first_ms)
    print("second_ms:", second_ms)
    print("first_agents:", [r.get("agent") for r in first.get("agent_results", [])])
    print("second_agents:", [r.get("agent") for r in second.get("agent_results", [])])
    print("summary:", second.get("final_summary", "")[:500])
else:
    print("Skipped. Set RUN_GRAPH_TEST = True to run this cell.")

## 10. Redis MONITOR Command

Redis logs do not show normal cache commands. To watch commands live, run this in a terminal while you run UI queries:

```powershell
docker compose exec redis redis-cli MONITOR
```

Look for `GET` on repeated queries and `SETEX` on cache misses. Stop it with `Ctrl+C`.

## 11. Cleanup Notebook Test Keys

This removes only `nb_redis_test:*` keys. It does not delete real app cache keys.

In [ ]:
deleted = delete_pattern(f"{TEST_PREFIX}:*")
print("deleted_test_keys:", deleted)
print("remaining_app_cache_keys:")
for pattern in ["information_agent:*", "knowledge_agent:*", "metadata_agent:*"]:
    print(pattern, len(redis_keys(pattern)))